# Pretrained Transformer: Airline Sentiment

## 1. Install Transformers

In [4]:
%pip install -q transformers

## 2. Import Libraries

In [5]:
import pandas as pd
from transformers import pipeline

## 3. Load the Dataset

In [6]:
df = pd.read_csv("Tweets.csv")
df = df[["text", "airline_sentiment"]].copy()
df.columns = ["text", "label"]

df.head()

,text,label
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


## 4. Prepare the Labels

In [7]:
df = df.dropna().copy()
df["text"] = df["text"].str.strip()
df = df[df["text"] != ""].copy()
df = df[df["label"].isin(["positive", "negative"])].copy()

df["true_label"] = df["label"].map({
    "positive": "POSITIVE",
    "negative": "NEGATIVE"
})

print(df["true_label"].value_counts())

true_label
NEGATIVE    9178
POSITIVE    2363
Name: count, dtype: int64


## 5. Take a Random Sample

In [8]:
sample = df.sample(n=200, random_state=42).copy()

print("Sample size:", len(sample))
print(sample["true_label"].value_counts())

Sample size: 200
true_label
NEGATIVE    160
POSITIVE     40
Name: count, dtype: int64


## 6. Load the Pretrained Model

In [9]:
classifier = pipeline("sentiment-analysis")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

## 7. Predict Sentiment

In [10]:
predictions = classifier(sample["text"].tolist(), truncation=True)

sample["predicted_label"] = [result["label"] for result in predictions]

## 8. Calculate Accuracy

In [11]:
correct = (sample["true_label"] == sample["predicted_label"]).sum()
accuracy = correct / len(sample)

print("Correct predictions:", correct, "out of", len(sample))
print("Accuracy:", round(accuracy * 100, 2), "%")

Correct predictions: 180 out of 200
Accuracy: 90.0 %


## 9. Show Five Example Rows

In [12]:
pd.set_option("display.max_colwidth", None)
sample[["text", "true_label", "predicted_label"]].head(5)

,text,true_label,predicted_label
11825,"@USAirways They charged me for a flight they Cancelled Flightled, unbelievable and unheard of",NEGATIVE,NEGATIVE
8105,@JetBlue great flight! Great view! :-) http://t.co/Yxn00pnOav,POSITIVE,POSITIVE
1279,"@united they're not, actually. gate agent was so rude. now standing in a line waiting for reFlight Booking Problems. missed the only flight to STI. awful.",NEGATIVE,NEGATIVE
14343,@AmericanAir No worries they called back 4 hrs Late Flightr while I was asleep and took an additional $200 fee. So by AA standards everything's gr8,NEGATIVE,NEGATIVE
3829,"@united thank you. There was one here a few months ago, but none now. Weird you don't have a club in one of the busiest airports in the US.",POSITIVE,NEGATIVE


## 10. What Does Attention Do?
Attention lets each word use relevant words elsewhere in the tweet as context, such as connecting “not” with “good.” A basic Bag-of-Words representation only counts words and does not represent their order or context-dependent relationships.

**Scope:** This is a small-sample evaluation of an existing model, not model training. The sample follows the dataset's class imbalance, so accuracy alone does not describe performance equally well for each class.